# Chapter 14: Advanced Panel Data Methods

Topics: fixed effects, random effects, Hausman test, clustered standard errors.

In [ ]:
import numpy as np
import pandas as pd
import wooldridge as woo
import matplotlib.pyplot as plt
import os
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS, compare

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## Data Setup

In [ ]:
df = woo.dataWoo('wagepan')
df = df.set_index(['nr', 'year'])
print(f"Panel dimensions: {df.index.get_level_values(0).nunique()} individuals × "
      f"{df.index.get_level_values(1).nunique()} years")
print(df.head())

## 14.1 Fixed Effects Estimation

In [ ]:
fe_model = PanelOLS.from_formula(
    'lwage ~ educ + black + hisp + exper + I(exper**2) + married + union + EntityEffects',
    data=df
).fit()
print(fe_model.summary)

## 14.2 Random Effects

In [ ]:
re_model = RandomEffects.from_formula(
    'lwage ~ educ + black + hisp + exper + I(exper**2) + married + union',
    data=df
).fit()
print(re_model.summary)

## Hausman Test (FE vs RE)

In [ ]:
# Compare via linearmodels
comparison = compare({'Pooled OLS': PooledOLS.from_formula('lwage ~ educ + exper + union', data=df).fit(),
                      'Fixed Effects': fe_model,
                      'Random Effects': re_model})
print(comparison)

## 14.4 Clustered Standard Errors

In [ ]:
fe_clustered = PanelOLS.from_formula(
    'lwage ~ exper + I(exper**2) + married + union + EntityEffects',
    data=df
).fit(cov_type='clustered', cluster_entity=True)
print("Fixed Effects with Clustered SEs:")
print(fe_clustered.summary)